# M21 · Deep-learning pickers: probability curves, pretrained weights and PocketQuake's picker

**Part 2 · week 3 · about 8 h core**

## 한국어 요약

딥러닝 위상 검출기(PhaseNet 계열)는 도달 시각을 바로 알려 주지 않고, 기록의 표본마다 P파와 S파가 도달했을 확률을 곡선으로 내놓습니다. 이 모듈에서는 먼저 임계값과 최소 간격을 써서 확률 곡선에서 피킹을 골라내는 함수를, 정답을 알고 있는 합성 곡선으로 직접 작성하고 점검합니다. 이어서 SeisBench로 서로 다른 자료로 학습한 두 가지 사전 학습 가중치를 2019년 7월 21일 상주 지진(ML 3.9) 기록에 적용하고, 두 결과를 여러분이 M20에서 직접 읽은 P파와 S파 도달 시각과 비교합니다. 그다음 PocketQuake의 피킹 단계(EQNet의 PhaseNet+)를 sgtl 도구로 실행해, 아무것도 바꾸지 않고 다시 실행한 결과가 처음과 같은지 확인한 뒤 확률 임계값 하나만 바꾸어 피킹 수가 어떻게 달라지는지 설명합니다. 마지막으로 PocketQuake 피킹 파일의 초동 극성 열을 읽어 단층면해(M51)에 쓸 초동 극성 표를 만들고, 동료가 건넨 피킹 파일로도 같은 작업을 해 봅니다.

## What you will be able to do

- Turn a pick-probability curve into picks with a threshold and a minimum gap, and test the function on made-up curves whose peaks are known
- Run two pretrained PhaseNet models with SeisBench on the Sangju records and measure how their P and S picks differ from your own manual picks of module M20
- Run PocketQuake's picking stage through sgtl twice without changes and once with a different probability threshold, and explain what changed in its pick files
- Read PocketQuake's pick files, including the first-motion polarity column, and turn them into a table of first motions for focal mechanisms

## Before you start

- Modules: M10, M20
- Reading: Mousavi & Beroza (2022), the whole article (a short overview of deep learning in seismology); Zhu & Beroza (2019), sections 1 to 3 (the network, its training data and its output); Woollam et al. (2022), the sections on models and on applying them to records (annotate and classify)
- Needs: stp, pocketquake
- Key terms (see the glossary): phase pick (위상 피킹, 도달 시각 읽기), deep-learning picker (딥러닝 위상 검출기), neural network (신경망), training data (학습 자료), model weights (모형 가중치), pretrained model (사전 학습 모형), inference (추론), PhaseNet (PhaseNet), STEAD (Stanford Earthquake Dataset; STEAD (스탠퍼드 지진 자료 집합)), sliding window (이동 창), amplitude normalization (진폭 정규화), accelerometer (가속도계), blinding (edge samples; 가장자리 표본 제외 (blinding)), annotations (SeisBench; 주석 결과 (SeisBench annotation)), pick probability (피킹 확률), pick threshold (피킹 임계값), local maximum (peak; 극댓값), false pick (오검출 피킹), systematic error (계통 오차), median absolute deviation (MAD; 중앙값 절대 편차), PocketQuake (지진 분석 도구 모음), cluster (PocketQuake; 클러스터 (PocketQuake)), PhaseNet+ (PhaseNet+), environment variable (환경 변수), deterministic computation (결정론적 계산), checksum (SHA-256 fingerprint; 체크섬), dry run (드라이 런, 시험 실행), first-motion polarity (초동 극성), focal mechanism (단층면해, 진원 메커니즘), SKHASH (단층면해 결정 프로그램), thread (스레드), origin time (진원시, 발생 시각), UTC (협정 세계시), P wave (P파), S wave (S파), trace (ObsPy Trace; 트레이스), stream (ObsPy Stream; 스트림)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M21.yaml](../../../.sgtl/criteria/M21.yaml).

## AI use

Follow [docs/ai_policy.md](../../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M21.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # folder of your STP download (module M03) inside the data folder
KMA_EVENT_ID = "2019009762"               # the ML 3.9 Sangju earthquake of 21 July 2019 in the KMA catalog
PICKS_CSV = "../picks/M20_manual_picks.csv"   # your manual picks of module M20 (in work/picks/)
WEIGHTS = {"stead": "2", "original": "2"}  # two pretrained PhaseNet weights (name: version); see Part 1
THRESHOLD = 0.3                           # smallest pick probability that may become a pick
MIN_GAP_S = 1.0                           # two picks of one phase closer than this (s): only the higher one counts
WINDOW_S = (-2.0, 40.0)                   # picks of this earthquake lie in this window (s after the origin time)
PQ_SLUG = "sangju_m21"                    # your PocketQuake cluster for this module (2-40 characters: a-z, 0-9, _)
PQ_MEM_GB = 8.0                           # memory cap of each PocketQuake run (GB); its picker needs about 3.5
PQ_MIN_PROB_CHANGED = 0.5                 # Part 5: the one PocketQuake setting you change (its default is 0.3)
FM_MIN_WEIGHT = 0.3                       # Part 6: smallest size of a polarity value that counts as a first motion
FM_MIN_PROB = 0.5                         # Part 6: smallest P-pick probability whose polarity counts
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M21")

import hashlib
import json
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, UTCDateTime, read

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)
# SeisBench looks for its weights in this folder; 'bin/sgtl-install seisbench' put the course's weights there.
os.environ.setdefault("SEISBENCH_CACHE_ROOT", str(st.pq.sgtl_home() / "seisbench"))
print("SeisBench weights folder:", os.environ["SEISBENCH_CACHE_ROOT"])

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M21-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up data whose answers are known from how they
were built (`st.reroll("M21-T1")` draws new ones), and on records of your own Sangju download and products of
your own PocketQuake runs; for those they never print the values they compare with.

This module needs three things from earlier work: your STP download of `sangju2019` (module M03), your manual
picks of module M20 (`work/picks/M20_manual_picks.csv`), and the course tools (`bin/sgtl-install tools` and
`bin/sgtl-install seisbench`, which put PocketQuake, EQNet and the PhaseNet weights under `~/sgtl`; read each
installer's dry run, then add `--apply`). Install those pinned weights before running the model cells; the
notebook expects them in the course cache. Everything runs on one CPU core: a deep-learning picker needs no graphics card for a few
hundred records.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M21.md`.

## Part 0 · Your data and your picks (about 15 min)

The earthquake comes from the KMA catalog that ships with the course (times in UTC; KMA, reference key
`kma_catalog`). Your manual picks of module M20 hold one row per **phase pick** (glossary `phase_pick`): the
station, the phase (P or S), the channel you picked on, the time in UTC, your uncertainty and, for P, the
first-motion **polarity** (glossary `polarity`: U for up, D for down, ? when unclear). The cell reads both and
then the three-component records of every station you picked, from the band (HH, HG or EL) you picked on.

In [ ]:
cat = st.data.kma_catalog()
event = cat.loc[cat["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN = UTCDateTime(event["origin_utc"].isoformat())
EVENT_FOLDER = ORIGIN.strftime("%Y%m%d%H%M%S")         # STP's name for this earthquake (UTC origin time)
print(f"KMA event {KMA_EVENT_ID}: origin {ORIGIN} (UTC), {float(event['latitude']):.2f} N, "
      f"{float(event['longitude']):.2f} E, depth {float(event['depth_km']):g} km, ML {float(event['ml']):.1f}")

SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / EVENT_FOLDER
if not SAC_DIR.is_dir():
    raise FileNotFoundError(f"There are no waveforms of this event in {SAC_DIR}. Fetch the {DATASET} data with STP "
                            "first (module M03), or set DATASET in the parameters cell to the folder you used.")
if not Path(PICKS_CSV).is_file():
    raise FileNotFoundError(f"{PICKS_CSV} is missing. This module compares the pickers with your own picks: finish "
                            "task M20-T5 of module M20 first (its picker saves the file).")
manual = pd.read_csv(PICKS_CSV)
manual["time"] = [UTCDateTime(t) for t in manual["time_utc"]]
print(f"\nyour manual picks ({len(manual)} rows, {manual['station'].nunique()} stations):")
print(manual[["station", "phase", "channel", "time_utc", "uncertainty_s", "polarity"]].to_string(index=False))

records = Stream()
for name, rows in manual.groupby("station"):
    band = str(rows["channel"].iloc[0])[:2]
    net, sta = name.split(".")
    records += read(str(SAC_DIR / band / f"*.{net}.{sta}.{band}?.sac"))
records.sort(keys=["network", "station", "channel"])
print(f"\n{len(records)} traces of {len(manual['station'].unique())} stations read, e.g.")
print(records[:3])

## Part 1 · What a deep-learning picker gives you (about 45 min)

A **deep-learning picker** (glossary `deep_learning_picker`) is a **neural network** (glossary
`neural_network`): a long chain of simple arithmetic steps whose millions of numbers, its **model weights**
(glossary `model_weights`), were set by training on **training data** (glossary `training_data`), records whose
arrivals analysts had picked. Running a trained network on new records is called **inference** (glossary
`inference`); nothing is learned then. A network together with weights that someone already trained is a
**pretrained model** (glossary `pretrained_model`). The same network with other weights is a different picker,
because it learned from other records and from other analysts' habits.

**PhaseNet** (glossary `phasenet`; Zhu & Beroza 2019) looks at 30 s of a three-component record at 100 samples
per second and gives, for every sample, three numbers between 0 and 1 that add up to 1: how likely it is that a P
wave, an S wave or neither (noise) arrives at that moment. Each of these series is a **pick-probability curve**
(glossary `pick_probability`). Mousavi & Beroza (2022) and Kong et al. (2019) give short overviews of such
networks; Münchmeyer et al. (2022) compare many of them on data from different regions.

**SeisBench** (Woollam et al. 2022) packages these networks with many sets of weights. Its `annotate` runs the
network along a whole record in a **sliding window** (glossary `sliding_window`): 30-s pieces that overlap, whose
outputs it glues together. Two details matter later:

- Each 30-s piece is scaled before it enters the network (**amplitude normalization**, glossary
  `amplitude_normalization`), so the size of the numbers does not matter: raw counts work as well as ground
  velocity in m/s. Scaling changes the size only, not the kind of motion. An **accelerometer** (glossary
  `accelerometer`) records ground acceleration, the rate at which ground velocity changes (its time
  derivative), and the same arrival looks different there: richer in high frequencies, with a sharper start.
  PhaseNet learned mostly from seismometer records of ground velocity, so on an accelerometer record it meets
  shapes it saw less often in training: its probabilities may come out lower, it may miss an arrival or pick
  noise, and its pick times may sit slightly off. Picks on accelerometer records (HG) deserve a closer look; you
  can also turn such a record into ground velocity first (response removal, module M10).
- The network sees too little record near the two ends of each piece, so SeisBench throws away the output there
  (**blinding**, glossary `blinding`). This is why the curves start a little later and end a little earlier than
  the record.

The course uses two sets of PhaseNet weights, pinned to fixed versions (`WEIGHTS` in the parameters cell):

- `stead`: trained by the SeisBench team on **STEAD** (glossary `stead`; Mousavi et al. 2019), about a million
  labeled records from many regions. It is also the picker that PocketQuake runs when nobody asks for another one.
- `original`: the weights that Zhu & Beroza (2019) trained on records from Northern California, converted to
  SeisBench.

The cell loads SeisBench and checks every weight file against its **checksum** (glossary `checksum`) in
`install/checksums.txt`, so you know you run exactly the course's weights. Then it prints what each model says
about itself.

In [ ]:
import torch

torch.set_num_threads(1)                         # inference on one core (shared-server rule)
try:
    import seisbench
    import seisbench.models as sbm
except ImportError as exc:
    raise ImportError(f"SeisBench cannot load its models in this environment ({exc}). Run 'sgtl doctor' in a "
                      "terminal and tell your mentor.") from None
print(f"SeisBench {seisbench.__version__}, PyTorch {torch.__version__}, threads used: {torch.get_num_threads()}")

pinned = {}
for line in st.data.repo_path("install", "checksums.txt").read_text(encoding="utf-8").splitlines():
    parts = line.split()
    if len(parts) == 3 and parts[0].startswith("phasenet/"):
        pinned[parts[0]] = parts[1]
models = {}
for name, version in WEIGHTS.items():
    for kind in ("json", "pt"):
        rel = f"phasenet/{name}.{kind}.v{version}"
        path = Path(seisbench.cache_model_root) / rel
        if not path.is_file():
            raise FileNotFoundError(f"{path} is missing. In a terminal: bin/sgtl-install seisbench --apply")
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        verdict = "matches the course's pin" if pinned.get(rel) == digest else "DOES NOT MATCH the pin: tell your mentor"
        print(f"{rel}: SHA-256 {digest[:16]}... {verdict}")
    models[name] = sbm.PhaseNet.from_pretrained(name, version_str=version)
    models[name].eval()
for name, model in models.items():
    print(f"\n{name}: {model.weights_docstring.strip().splitlines()[0]}")
    print(f"  component order {model.component_order}, normalization {model.norm!r}, "
          f"settings that come with the weights: {model.default_args}")

Now run the `stead` model on the station you picked first. Compare the start and end of the record with those of
the probability curves, and check that the three curves add up to 1.

In [ ]:
first_station = manual.sort_values("time").iloc[0]["station"]
one = records.select(network=first_station.split(".")[0], station=first_station.split(".")[1])
curves = models["stead"].annotate(one)
print(curves)
print(f"\nrecord:   {one[0].stats.starttime} to {one[0].stats.endtime} ({one[0].stats.npts} samples)")
p_curve = curves.select(channel="*_P")[0]
s_curve = curves.select(channel="*_S")[0]
print(f"P curve:  {p_curve.stats.starttime} to {p_curve.stats.endtime} ({p_curve.stats.npts} samples)")
total = np.sum([tr.data for tr in curves], axis=0)
print(f"P + S + noise lies between {total.min():.4f} and {total.max():.4f} at every sample")

z = one.select(component="Z")[0]
fig, axes = plt.subplots(2, 1, figsize=(7.0, 4.6), sharex=True)
axes[0].plot(z.times(reftime=ORIGIN), z.data - np.mean(z.data), color="k", lw=0.5)
axes[0].set_ylabel(f"{z.stats.channel} (counts)")
axes[1].plot(p_curve.times(reftime=ORIGIN), p_curve.data, color="tab:blue", lw=0.9, label="P")
axes[1].plot(s_curve.times(reftime=ORIGIN), s_curve.data, color="tab:red", lw=0.9, label="S")
for _, row in manual[manual["station"] == first_station].iterrows():
    for ax in axes:
        ax.axvline(row["time"] - ORIGIN, color="0.5", ls="--", lw=0.8)
axes[1].set_ylabel("Probability")
axes[1].set_xlabel("Time after the origin (s)")
axes[1].set_xlim(-5.0, 30.0)
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8, title=f"{first_station}, stead weights "
           "(dashed: your picks)", title_fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "probability_curves.png", dpi=150)

A **local maximum** (glossary `local_maximum`) of a curve is a sample higher than both of its neighbours. A
picker turns a curve into picks in three steps: it looks for the local maxima, keeps those that reach a **pick
threshold** (glossary `pick_threshold`), and does not allow two picks of one phase closer than a minimum gap.
Each step has a reason. A curve can have small humps on the shoulder of one arrival, and without the gap each hump
would be another pick. A threshold too low lets noise through: a **false pick** (glossary `false_pick`) is a
pick where no arrival is. A threshold too high, or a gap too long, loses real arrivals.

### M21-T1 · From a probability curve to picks · core · about 60 min

**Goal:** write `picks_from_probability(prob, delta, threshold, min_gap_s)`: the picks in one pick-probability
curve.

**Given:** the curves of Part 1; a curve is a NumPy array with one probability per sample.

**Contract:**
- Inputs: `prob`, a 1-D NumPy array of probabilities (0 to 1), one per sample; `delta`, the sample spacing in s;
  `threshold`; `min_gap_s`, in s.
- A peak is a sample higher than both of its neighbours (a local maximum); the first and the last sample are
  never peaks.
- A pick is a peak whose probability is at least `threshold`.
- When two picks lie closer than `min_gap_s` to each other, only the higher one counts. Decide from the highest
  pick down: a pick stays when no higher pick that stays lies closer than `min_gap_s` to it.
- Output: a tuple `(times_s, values)` of two 1-D NumPy float arrays of the same length, sorted by time:
  `times_s`, the pick times in seconds after the first sample (sample number × `delta`), and `values`, the
  probabilities at those samples.
- `prob` itself must not change.
- Example: `prob = [0, 0.2, 0.6, 0.3, 0.7, 0.1, 0, 0.4, 0]`, `delta = 0.5`, `threshold = 0.35` and
  `min_gap_s = 1.5` give `(array([2. , 3.5]), array([0.7, 0.4]))`: the peaks at 1.0 s (0.6) and 2.0 s (0.7)
  lie 1.0 s apart, so only the higher stays; the peak at 3.5 s is 1.5 s from the one at 2.0 s, which is not
  closer than 1.5 s.

**Done when** `st.check("M21-T1", func=picks_from_probability)` passes on 150 made-up curves (sample spacings
from 4 to 25 ms, thresholds from 0.1 to 0.6, gaps of 8 to 120 samples; bumps closer and farther apart than the
gap; curves that still rise at the last sample or fall from the first), compared with an independent peak
finder that your function may not use.

**Stuck?** `st.hint("M21-T1", 1)`

In [ ]:
# Before you code: in the example of the Contract, what do you get with min_gap_s = 0.5 instead, and what with
# threshold = 0.65? The P curve of Part 1 covers the whole 500-s record (the figure shows only 35 s of it, from 5 s
# before to 30 s after the origin): how many picks do you expect on it with threshold 0.05 and no gap at all, and
# how many with 0.3 and a 1-s gap? What would a very long gap (say 60 s) do to two earthquakes a few seconds apart?
st.predict("M21-T1", """

""")

In [ ]:
def picks_from_probability(prob, delta, threshold, min_gap_s):
    """Picks in one pick-probability curve: peaks at or above a threshold, never two closer than min_gap_s.

    prob      : 1-D numpy array, one probability per sample (it is not changed)
    delta     : sample spacing (s)
    threshold : smallest probability of a pick
    min_gap_s : two picks closer than this (s) cannot both stay; the higher one stays
    returns   : (times_s, values), two 1-D float arrays sorted by time; times_s counted from the first sample
    """
    # Step 1: the peaks: samples higher than the sample before and the sample after (never the two end samples).
    # Step 2: keep the peaks at or above the threshold.
    # Step 3: visit them from the highest down, and keep one only when no kept pick lies closer than min_gap_s.
    # Step 4: sort what you kept by time; give back the times in seconds and the probabilities.
    # YOUR CODE HERE
    raise NotImplementedError("M21-T1: write your code here")

In [ ]:
st.check("M21-T1", func=picks_from_probability)

The whole 500-s P curve of Part 1: first where it rises above 0.1 and how high it gets there, then your function
on it with a few thresholds and gaps.

In [ ]:
above = p_curve.data >= 0.1
edges = np.flatnonzero(np.diff(above.astype(int)) == 1) + 1
print(f"the P curve rises above 0.1 at {edges.size} places:")
for k in edges:
    stop = k + int(np.argmax(~above[k:])) if (~above[k:]).any() else p_curve.stats.npts
    print(f"  {p_curve.times(reftime=ORIGIN)[k]:7.2f} s after the origin, highest value {p_curve.data[k:stop].max():.3f}")
print()
for thr, gap in ((0.05, 0.0), (0.1, 0.0), (0.3, 0.0), (0.3, MIN_GAP_S), (0.6, MIN_GAP_S)):
    secs, probs = picks_from_probability(p_curve.data, p_curve.stats.delta, thr, gap)
    after = [round(float(p_curve.stats.starttime + x - ORIGIN), 2) for x in secs]
    print(f"threshold {thr:4.2f}, gap {gap:3.1f} s: {len(secs):3d} picks at (s after the origin) {after[:8]}"
          + (" ..." if len(after) > 8 else ""))

**Explain (3-5 sentences):** On the real P curve, how many picks did your function give for each threshold and
gap, and which of them are the P of the mainshock (and what are the others)? Why does a curve that still rises at
the last sample give no pick there, and why is that the safe choice for a curve that a window cuts off? What does
the minimum gap protect you from, and what can it cost you?

## Part 2 · Picks of two pretrained models (about 75 min)

SeisBench's `annotate` returns a Stream (module M10; glossary `stream`) of probability curves, one **trace**
(glossary `trace`) per station and phase: SeisBench calls it the **annotations** (glossary
`seisbench_annotation`). Each curve's channel name ends with `_P`, `_S` or `_N` (noise), and each curve keeps its
own start time. Task 2 turns such a Stream into one pick per station and phase.

### M21-T2 · The best pick of every station · core · about 60 min

**Goal:** write `best_picks(annotations, threshold, min_gap_s, start, end)`: for every station, the most probable
P pick and the most probable S pick inside a time window.

**Given:** your `picks_from_probability` (task 1) and the annotations of a SeisBench model.

**Contract:**
- Inputs: `annotations`, the Stream that a SeisBench model's `annotate` returns (curves whose channel names end
  with `_P`, `_S` or `_N`); `threshold` and `min_gap_s` as in task 1; `start` and `end`, two obspy
  `UTCDateTime` moments: the window in which picks may lie (both ends included).
- For every station and for P and S: the picks of your `picks_from_probability` on the whole curve, then only
  those between `start` and `end`, then the one with the highest probability. Noise curves give no picks.
- The time of sample k of a curve is that curve's own start time plus k × `delta`.
- Output: a pandas DataFrame with the columns `station` (network.station, for example `KS.ZZA1`), `phase` (`P` or
  `S`), `time` (obspy `UTCDateTime`) and `probability`; one row per station and phase that has a pick inside the
  window, sorted by station, then phase.
- Example: a station whose P curve has picks at 02:04:25.1 (0.9), 02:04:40.0 (0.95) and 02:10:51.3 (0.97),
  with the window 02:04:16 to 02:04:58, gives the row with 02:04:40.0 and 0.95.

**Done when** `st.check("M21-T2", func=best_picks, dataset=DATASET)` passes: on 8 made-up annotation Streams
(1-3 stations, curves with their own start times, peaks inside and outside the window, noise curves with high
peaks), and on records of 3 random HH stations of your download, which the check annotates itself with the
`stead` weights: your rows must lie on the model's curves, inside the window and above the threshold, and agree
with SeisBench's own peak finder for the clear picks. It never prints those values.

**Stuck?** `st.hint("M21-T2", 1)`

In [ ]:
# Before you code: the two models learned from different records and different analysts. At your stations, do you
# expect their P picks to lie within one sample (0.01 s) of each other, within 0.05 s, or farther apart? And their
# S picks? Which model do you expect to agree better with your own picks, and why?
st.predict("M21-T2", """

""")

In [ ]:
def best_picks(annotations, threshold, min_gap_s, start, end):
    """The most probable P and S pick of every station inside [start, end].

    annotations     : obspy Stream from a SeisBench model's annotate (channel names end with _P, _S or _N)
    threshold       : smallest probability of a pick
    min_gap_s       : the minimum gap of picks_from_probability (s)
    start, end      : obspy UTCDateTime limits of the window (both included)
    returns         : pandas DataFrame with columns station, phase, time (UTCDateTime), probability,
                      sorted by station and phase
    """
    # Step 1: go through the P and S curves (their channel names end with _P or _S); skip the noise curves.
    # Step 2: run your picks_from_probability on each curve; turn its times into UTC with the curve's start time.
    # Step 3: keep the picks between start and end, and of these the most probable one.
    # Step 4: one row per station and phase; sort by station, then phase.
    # YOUR CODE HERE
    raise NotImplementedError("M21-T2: write your code here")

In [ ]:
st.check("M21-T2", func=best_picks, dataset=DATASET)

Both models on all your stations, with the window of the parameters cell. The table goes to
`results/M21_model_picks.csv`.

In [ ]:
window = (ORIGIN + WINDOW_S[0], ORIGIN + WINDOW_S[1])
per_model = []
for name, model in models.items():
    rows = best_picks(model.annotate(records), THRESHOLD, MIN_GAP_S, *window)
    rows.insert(0, "model", name)
    per_model.append(rows)
model_picks = pd.concat(per_model, ignore_index=True)
shown = model_picks.assign(after_origin_s=[round(t - ORIGIN, 3) for t in model_picks["time"]])
print(shown.drop(columns="time").to_string(index=False))
model_picks.assign(time=model_picks["time"].astype(str)).to_csv(Path(RESULT_DIR) / "M21_model_picks.csv",
                                                                 index=False)

**Explain (3-5 sentences):** How many of your stations got a P and an S pick from each model, and how sure were
the models (the range of the probabilities)? At how many stations did the two models' P picks differ by more than
one sample, and in which direction? Why must the time of a pick come from the start time of its own curve, and
what would go wrong if you took the record's start time instead?

## Part 3 · Your picks against the models' picks (about 45 min)

To compare two sets of picks you match them: the same station and phase (and, for many earthquakes, the same
earthquake). Some picks have no partner, and those matter as much as the differences: a model may miss an
arrival you saw, or find one you did not pick.

Two kinds of difference behave differently. A **systematic error** (glossary `systematic_error`) has the same
sign and size at every station, for example a model that always picks 0.03 s late; averaging more stations does
not remove it. A random scatter changes sign from station to station. The median of the differences shows the
first, and the **median absolute deviation** (glossary `median_absolute_deviation`, MAD) shows the second without
letting one wild pick dominate.

### M21-T3 · Matching two sets of picks · core · about 45 min

**Goal:** write `compare_pick_sets(a, b)`: one table that matches the picks of two pick tables and gives the time
difference of every matched pair.

**Given:** pandas `merge` (an "outer" join keeps rows that have no partner); your manual picks and the models'
picks of Part 2.

**Contract:**
- Inputs: two pick tables `a` and `b` (pandas DataFrames) with the columns `station`, `phase` and `time` (obspy
  `UTCDateTime`), maybe `event_id`, and maybe other columns. The key of a pick is (`event_id`, `station`,
  `phase`) when both tables have an `event_id` column, otherwise (`station`, `phase`).
- A table that holds one key twice is an error: raise a `ValueError`.
- Output: a DataFrame with one row per key found in `a` or in `b`: the key columns, `time_a` and `time_b` (the
  times, or `None` where that table has no such pick), `diff_s` = `time_b` − `time_a` in seconds (NaN unless both
  tables have the pick) and `status`: `"both"`, `"only_a"` or `"only_b"`; sorted by the key columns.
- `a` and `b` must not change.
- Example (made-up network XX, stations and times on 2030-01-01): `a` with XX.ST01 P at 10:00:03.250 and S at
  10:00:05.400, `b` with XX.ST01 P at 10:00:03.265 and XX.ST02 P at 10:00:04.100 give three rows: XX.ST01 P
  (both, +0.015 s), XX.ST01 S (only_a, NaN), XX.ST02 P (only_b, NaN).

**Done when** `st.check("M21-T3", func=compare_pick_sets)` passes on 40 pairs of made-up tables (with and
without `event_id`, shuffled rows, extra columns, picks in only one table, a few large differences) and on 3 tables
that hold a key twice.

**Stuck?** `st.hint("M21-T3", 1)`

In [ ]:
# Before you code: you will compare your manual picks (a) with each model (b). Which sign of diff_s means that the
# model picked later than you? Do you expect one model's P differences to scatter around zero, or to sit mostly on
# one side? For a location (module M31), which hurts more: a scatter of 0.02 s or an offset of 0.03 s that is the
# same at every station?
st.predict("M21-T3", """

""")

In [ ]:
def compare_pick_sets(a, b):
    """Match the picks of two pick tables by their key and give the time differences b - a.

    a, b    : pandas DataFrames with columns station, phase, time (obspy UTCDateTime) and maybe event_id; not changed
    returns : DataFrame with the key columns, time_a, time_b (None when missing), diff_s (s; NaN unless both)
              and status ("both", "only_a", "only_b"), sorted by the key
    raises  : ValueError when a key appears twice in a or in b
    """
    # Step 1: the key: event_id (only when both tables have that column), station and phase.
    # Step 2: stop with a ValueError when a key appears twice in either table.
    # Step 3: an outer join of the two tables on the key, keeping each table's time under its own name.
    # Step 4: status, empty places as None, diff_s in seconds for keys in both; sort by the key.
    # YOUR CODE HERE
    raise NotImplementedError("M21-T3: write your code here")

In [ ]:
st.check("M21-T3", func=compare_pick_sets)

Your picks (a) against each model (b): every pair, then the median and the MAD per model and phase. The figure
shows the differences against the time of your P pick (a stand-in for the distance).

In [ ]:
mine = manual[["station", "phase", "time"]]
summary, matched = [], {}
for name in models:
    table = compare_pick_sets(mine, model_picks[model_picks["model"] == name])
    matched[name] = table
    print(f"\n{name}:")
    print(table.assign(time_a=table["time_a"].astype(str)).drop(columns="time_b").round(3).to_string(index=False))
    for phase, part in table[table["status"] == "both"].groupby("phase"):
        d = part["diff_s"].to_numpy()
        summary.append({"model": name, "phase": phase, "matched": d.size, "median_s": np.median(d),
                        "mad_s": np.median(np.abs(d - np.median(d))), "min_s": d.min(), "max_s": d.max()})
summary = pd.DataFrame(summary)
print("\n" + summary.round(3).to_string(index=False))
summary.to_csv(Path(RESULT_DIR) / "M21_pick_differences.csv", index=False)

p_time = {r["station"]: r["time"] - ORIGIN for _, r in manual[manual["phase"] == "P"].iterrows()}
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.4), sharey=True)
for ax, phase in zip(axes, ("P", "S")):
    for name, color in zip(models, ("tab:blue", "tab:orange")):
        part = matched[name][(matched[name]["phase"] == phase) & (matched[name]["status"] == "both")]
        ax.plot([p_time[s] for s in part["station"]], part["diff_s"], "o", color=color, ms=4, label=name)
    ax.axhline(0.0, color="0.5", lw=0.8)
    ax.set_title(f"{phase} picks", fontsize=9)
    ax.set_xlabel("Your P pick (s after the origin)")
axes[0].set_ylabel("Model minus your pick (s)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "model_minus_manual.png", dpi=150)

**Explain (3-5 sentences):** For each model and phase, how many of your picks did it match, and what were the
median and the MAD of the differences? Is one model's P systematically early or late compared with yours (compare
its median with its MAD), and what in its training data could make it so? Which model would you trust for S, and
why do the S differences scatter more than the P differences?

## Part 4 · PocketQuake's picker, run three times (about 90 min)

**PocketQuake** (glossary `pocketquake`) runs picking as one stage of its chain (stations, waveforms, picking,
location, relocation, mechanisms). Through `st.pq` its picker is **PhaseNet+** (glossary `phasenet_plus`; Zhu et
al. 2025) from EQNet (Zhu et al. 2022): a network of the PhaseNet family that also gives every P pick a
first-motion polarity and an amplitude. PocketQuake works on a **cluster** (glossary `pocketquake_cluster`): your
dataset registered under a short name, `PQ_SLUG`. Its results go to their own folder inside your PocketQuake copy.

A production tool must be tested like your own code. Two runs, in this order:

1. **An unchanged rerun.** Is the result **deterministic** (glossary `deterministic_computation`): the same, byte
   for byte, when nothing changed? If not, every later comparison mixes your change with this noise.
2. **One setting changed.** PocketQuake reads the threshold of its PhaseNet+ picker from an **environment
   variable** (glossary `environment_variable`) called `PNPLUS_MIN_PROB` (default 0.3). A program started from
   Python inherits the environment of the Python process, so you set the variable here, run, and remove it again.

Each run is a heavy job, so `st.pq` first does a **dry run** (glossary `dry_run`) that only prints what it would
do; `apply=True` runs it, on one core, under `nice` and a memory cap. Every run is logged with the SHA-256
fingerprints of the pick files it wrote in `work/.sgtl/runs.jsonl`. A run overwrites the pick folder, so the cells
copy it after each run.

### M21-T4 · PocketQuake's pick files, three runs · core · about 90 min

**Goal:** write `read_pocketquake_picks(folder)`, run PocketQuake's picking stage three times (first run,
unchanged rerun, one setting changed), and report the three runs in a dictionary `pq_runs`.

**Given:** the run cells below (read each dry run before you apply it) and the pick files they copy to
`results/pq_picks/`.

**Contract (the reader):**
- Input: `folder` (a `str` or a `pathlib.Path`): a picks folder of PocketQuake, with one file
  `<event id>_picks.csv` per earthquake. Its columns are `Event_ID`, `Network`, `Station`, `Phase`, `Time` (ISO
  text in UTC) and `Probability`, and, from a picker that measures them, `Polarity` and `Amplitude`. Other files in
  the folder are not pick files.
- Output: one pandas DataFrame with every pick of every `*_picks.csv` file and the columns `event_id` (text, such
  as `"20000101000000"`), `station` (network.station), `phase`, `time` (obspy `UTCDateTime`), `probability`
  (float) and, only when the files have a `Polarity` column, `polarity` (float, NaN for S picks). Never make up a
  column the files do not have. Other columns may stay.

**Contract (the report):** `pq_runs = {"reader": read_pocketquake_picks, "slug": PQ_SLUG, "first": <copy of the
first run>, "rerun": <copy of the rerun>, "changed": <copy of the changed run>, "min_prob_changed":
PQ_MIN_PROB_CHANGED, "fingerprints": {"first": <fingerprints of the first copy>, "rerun": <fingerprints of the
rerun copy>}}`, where the fingerprints of a copy are a dict from the name of every pick file in it (such as
`"20000101000000_picks.csv"`) to the SHA-256 of that file's bytes (glossary `checksum`; 64 hex letters, as
`hexdigest()` writes them).

**Done when** `st.check("M21-T4", answer=pq_runs)` passes: the check runs your reader on 6 made-up pick folders
(with and without polarities, with other files), then reads your three copies: it recomputes every fingerprint you
give from the files, the copies must be among the products logged for runs of your cluster, the rerun must be a run
of its own, every pick of the changed run must reach `min_prob_changed` and every pick of the other two the default
of 0.3. It never prints pick values or fingerprints.

**Stuck?** `st.hint("M21-T4", 1)`

In [ ]:
# Before you run: will the unchanged rerun write the same pick files, byte for byte? With PocketQuake's threshold
# raised from 0.3 to PQ_MIN_PROB_CHANGED, will picks disappear, appear, or move to other times, and will the
# mainshock or the smaller earthquakes change most?
st.predict("M21-T4", """

""")

In [ ]:
def read_pocketquake_picks(folder):
    """All picks of a PocketQuake picks folder (<event id>_picks.csv files) in one table.

    folder  : str or pathlib.Path of the folder
    returns : pandas DataFrame with event_id (str), station (network.station), phase, time (UTCDateTime),
              probability (float) and, only when the files have a Polarity column, polarity (float)
    """
    # Step 1: the pick files only: names ending in _picks.csv, in name order.
    # Step 2: read them with pandas, with Event_ID kept as text (it is a name, not a number), and stack them.
    # Step 3: build the columns of the Contract; add polarity only when the files have a Polarity column.
    # YOUR CODE HERE
    raise NotImplementedError("M21-T4: write your code here")

Run 1. First register your dataset as a PocketQuake cluster (once; read the dry run, then apply), then run the
stages stations, waveforms and picking (about 2 minutes on one core).

In [ ]:
cluster_file = st.pq.pipeline_dir() / "pipeline" / "clusters" / f"{PQ_SLUG}.py"
if cluster_file.is_file():
    print(f"Cluster {PQ_SLUG!r} is registered already ({cluster_file.name}); nothing to stage.")
else:
    st.pq.stage(DATASET, PQ_SLUG)                             # dry run: read what it would do

In [ ]:
if not cluster_file.is_file():
    assert st.pq.stage(DATASET, PQ_SLUG, apply=True) == 0
st.pq.run(PQ_SLUG, "stations", "picking", cores=1, mem_gb=PQ_MEM_GB)     # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "stations", "picking", cores=1, mem_gb=PQ_MEM_GB, apply=True) == 0
PQ_PICKS = st.pq.pipeline_dir() / "pipeline" / "runs" / PQ_SLUG / "picks"
COPIES = Path(RESULT_DIR) / "pq_picks"


def keep_copy(label):
    """Copy PocketQuake's picks folder to results/pq_picks/<label> (the next run overwrites the original)."""
    target = COPIES / label
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(PQ_PICKS, target)
    print(f"copied {len(list(target.glob('*_picks.csv')))} pick files to {target}")
    return target


first_dir = keep_copy("first")

Run 2: the picking stage again, nothing changed.

In [ ]:
assert st.pq.run(PQ_SLUG, "picking", "picking", cores=1, mem_gb=PQ_MEM_GB, apply=True) == 0
rerun_dir = keep_copy("rerun")

Run 3: one setting changed. This is the line of PocketQuake's settings (in your copy) that reads the threshold:

In [ ]:
config_lines = (st.pq.pipeline_dir() / "pipeline" / "config.py").read_text(encoding="utf-8").splitlines()
print("\n".join(f"config.py:{n}: {line}" for n, line in enumerate(config_lines, 1) if "PNPLUS_MIN_PROB" in line))

In [ ]:
os.environ["PNPLUS_MIN_PROB"] = str(PQ_MIN_PROB_CHANGED)      # PocketQuake reads it when the run starts
try:
    status = st.pq.run(PQ_SLUG, "picking", "picking", cores=1, mem_gb=PQ_MEM_GB, apply=True)
finally:
    del os.environ["PNPLUS_MIN_PROB"]                         # every later run gets the default again
assert status == 0
changed_dir = keep_copy(f"min_prob_{PQ_MIN_PROB_CHANGED:g}")

The run log: one line per run, with the fingerprints of the files it wrote.

In [ ]:
log_lines = st.data.repo_path("work", ".sgtl", "runs.jsonl").read_text(encoding="utf-8").splitlines()
for line in log_lines[-3:]:
    run = json.loads(line)
    print(run["ts"], f"{run['stage_from']} -> {run['through']}", f"{run['seconds']:.0f} s", run["products"])

In [ ]:
# Fill in pq_runs from your three runs. Compute the fingerprints from the pick files in your two copies (first and
# rerun), with hashlib, after the copy cells: the check recomputes every one.
# YOUR CODE HERE
raise NotImplementedError("M21-T4: write your code here")
print({k: (v.__name__ if callable(v) else v) for k, v in pq_runs.items() if k != "fingerprints"})

In [ ]:
st.check("M21-T4", answer=pq_runs)

The two copies file by file, from your fingerprints, and what changed between the runs, with your
`compare_pick_sets` of task 3 (the key now includes the earthquake):

In [ ]:
prints = pq_runs["fingerprints"]
same_files = sorted(n for n in prints["first"] if prints["rerun"].get(n) == prints["first"][n])
print(f"pick files with the same SHA-256 in the first run and the rerun: {len(same_files)} of "
      f"{len(prints['first'])} (the rerun has {len(prints['rerun'])})")
first_run = read_pocketquake_picks(first_dir)
print(f"first run: {len(first_run)} picks of {first_run['event_id'].nunique()} earthquakes, "
      f"{first_run['station'].nunique()} stations; probabilities {first_run['probability'].min():.3f} to "
      f"{first_run['probability'].max():.3f}")
for label, folder in (("rerun", rerun_dir), ("changed", changed_dir)):
    table = compare_pick_sets(first_run, read_pocketquake_picks(folder))
    moved = int(((table["status"] == "both") & (table["diff_s"].abs() > 0.0)).sum())
    print(f"\nfirst run against the {label} run: {table['status'].value_counts().to_dict()}, moved picks: {moved}")
    print(table.groupby(["event_id", "status"]).size().unstack(fill_value=0).to_string())

**Explain (3-5 sentences):** Was the unchanged rerun identical, and what makes PocketQuake's picking repeat (or
not)? With the threshold at `PQ_MIN_PROB_CHANGED`, how many picks were lost or gained per earthquake, did any pick
move, and why did the mainshock change least? PocketQuake keeps at most one P and one S per station, inside a
window around the expected arrival times: how does that rule shape the change in the pick counts?

## Part 5 · First motions from a pick file (about 60 min)

A **focal mechanism** (glossary `focal_mechanism`; module M51) can be found from the first-motion polarities of
P at many stations: programs such as HASH and **SKHASH** (glossary `skhash`; Hardebeck & Shearer 2002; Skoumal
et al. 2024) search for the fault orientations that fit the pattern of ups and downs. Deep-learning pickers can
read polarities too (Ross et al. 2018; Uchide 2020). In PocketQuake's pick files the `Polarity` column holds a
number from −1 to 1: its sign is the direction (positive up), and its size says how sure the picker is; PocketQuake
drops polarities smaller than 0.3 and polarities of P picks with a probability below 0.5.

A colleague hands you a pick file of the same earthquake and a helper that turns pick files into first motions
for a mechanism program.

### M21-T5 · First motions from a pick table · core · about 60 min

**Goal:** write `first_motions(picks, min_weight, min_prob)`: the first motion (U, D or ?) of every P pick of a
pick table that your reader of task 4 made.

**Given:** the colleague's file and helper (the cell below), PocketQuake's rule above, and your PocketQuake picks.

**Contract:**
- Inputs: `picks`, a pick table as your `read_pocketquake_picks` returns it (columns `station`, `phase`,
  `probability`, maybe `event_id`, maybe `polarity`); `min_weight` and `min_prob`.
- Output: a pandas DataFrame with one row per P pick (S picks give no rows), with `event_id` (only when `picks` has
  it), `station` and `first_motion`: `"U"` when the polarity is at least `min_weight`, `"D"` when it is at most
  −`min_weight`, and `"?"` otherwise or when the pick's probability is below `min_prob`; sorted by `event_id`
  (when present), then `station`. `picks` must not change.
- Example: with `min_weight = 0.3` and `min_prob = 0.5`, P picks with (polarity, probability) of (0.8, 0.9),
  (−0.3, 0.7), (0.2, 0.9) and (0.9, 0.4) give U, D, ?, ?.

**Done when** `st.check("M21-T5", func=first_motions)` passes on 40 made-up pick tables (with and without
`event_id`; polarities on both sides of ±`min_weight` and exactly on it; probabilities on both sides of `min_prob`;
missing polarity values; S picks) and on 3 more made-up tables.

**Stuck?** `st.hint("M21-T5", 1)`

In [ ]:
# The colleague's pick file and helper. Run this cell (it writes the file and defines the helper); do not edit it.
# The file holds the colleague's picks of the mainshock, made with PhaseNet and the 'stead' weights, which is
# what PocketQuake's picking stage runs when it is started without a picker option (picker_weights='stead').
colleague_dir = Path(RESULT_DIR) / "colleague_picks"
colleague_dir.mkdir(exist_ok=True)
theirs = model_picks[model_picks["model"] == "stead"]
pd.DataFrame({"Event_ID": EVENT_FOLDER, "Network": theirs["station"].str.split(".").str[0],
              "Station": theirs["station"].str.split(".").str[1], "Phase": theirs["phase"],
              "Time": theirs["time"].astype(str), "Probability": theirs["probability"].round(3)}
             ).to_csv(colleague_dir / f"{EVENT_FOLDER}_picks.csv", index=False)


def colleague_first_motions(folder):
    """The colleague's helper: first motions (U = up, D = down) of the P picks of a PocketQuake picks folder."""
    picks = pd.concat([pd.read_csv(f, dtype={"Event_ID": str}) for f in sorted(Path(folder).glob("*_picks.csv"))])
    p = picks[picks["Phase"] == "P"]
    weight = p["Polarity"] if "Polarity" in p.columns else pd.Series(0.0, index=p.index)
    return pd.DataFrame({"station": p["Network"] + "." + p["Station"],
                         "first_motion": np.where(weight.fillna(0.0) >= 0.0, "U", "D")})


print(f"wrote {colleague_dir / f'{EVENT_FOLDER}_picks.csv'}")

In [ ]:
# Before you code: what will colleague_first_motions give for the colleague's folder: how many U, D and ? (look at
# the columns of the file)? What would a mechanism program (module M51) make of that table? How could you tell from
# a pick file alone whether its picker measured first motions?
st.predict("M21-T5", """

""")

Now measure: the columns of the colleague's file, what the helper makes of it, and what PocketQuake's own code says
about pickers and polarities (lines printed from your copy of PocketQuake).

In [ ]:
print("columns of the colleague's file:", pd.read_csv(next(colleague_dir.glob("*_picks.csv"))).columns.tolist())
print("the colleague's helper:", colleague_first_motions(colleague_dir)["first_motion"].value_counts().to_dict())
pipeline_code = st.pq.pipeline_dir() / "pipeline"
for rel, words in (("config.py", ("SEISBENCH_MODELS =", "EQNET_MODELS =", "picker_weights: str")),
                   ("clusters/_base.py", ("picker_weights=",)),
                   ("core/focal_mechanism.py", ('"Polarity" not in', "phasenet_plus picking run"))):
    for number, line in enumerate((pipeline_code / rel).read_text(encoding="utf-8").splitlines(), 1):
        if any(w in line for w in words):
            print(f"{rel}:{number}: {line.strip()}")

In [ ]:
def first_motions(picks, min_weight, min_prob):
    """First motions (U, D or ?) of the P picks of a pick table made by read_pocketquake_picks.

    picks      : pandas DataFrame with station, phase, probability, maybe event_id, and polarity; not changed
    min_weight : smallest size of a polarity value that counts as a first motion
    min_prob   : smallest P-pick probability whose polarity counts
    returns    : DataFrame with event_id (when present), station and first_motion, sorted
    raises     : ValueError when the table has no polarity column
    """
    # Step 1: no polarity column means no polarity at all: raise a ValueError (never guess).
    # Step 2: the P picks only.
    # Step 3: U when the polarity reaches min_weight, D when it reaches -min_weight, else a question mark (also for
    #         NaN, and for every pick whose probability is below min_prob).
    # Step 4: the key columns and first_motion, sorted.
    # YOUR CODE HERE
    raise NotImplementedError("M21-T5: write your code here")

In [ ]:
st.check("M21-T5", func=first_motions)

The 3 more tables of the check had no `polarity` column, and your function had to stop on them with a
`ValueError`: a table without polarities holds no first motions at all.

Your function on the colleague's folder and on your own first PocketQuake run; then PhaseNet+'s first motions of
the mainshock next to yours of module M20.

In [ ]:
try:
    first_motions(read_pocketquake_picks(colleague_dir), FM_MIN_WEIGHT, FM_MIN_PROB)
    print("your function gave first motions for the colleague's file")
except ValueError as exc:
    print("your function stopped on the colleague's file:", exc)

motions = first_motions(first_run, FM_MIN_WEIGHT, FM_MIN_PROB)
print("\nPhaseNet+ first motions, all earthquakes of your first run:")
print(motions.groupby(["event_id", "first_motion"]).size().unstack(fill_value=0).to_string())
side = (manual[manual["phase"] == "P"][["station", "polarity"]].rename(columns={"polarity": "yours"})
        .merge(motions[motions["event_id"] == EVENT_FOLDER][["station", "first_motion"]], on="station", how="left"))
side["agree"] = side["yours"] == side["first_motion"]
print("\nmainshock: your first motions (M20) and PhaseNet+'s")
print(side.to_string(index=False))
print(f"agree at {int(side['agree'].sum())} of {len(side)} stations")

**Explain (3-5 sentences):** What did the colleague's helper give for the colleague's file, and why? Which of
PocketQuake's picker settings give no polarities (quote the lines you printed), and what does its mechanism stage
(module M51) do with a pick file without them? What would a mechanism program have found if it had been given the
colleague's table? At how many of your stations does PhaseNet+ agree with your first motions, and where you
disagree, how would you decide which one to trust?

## What you made, and what comes next

- `results/M21_model_picks.csv`: the picks of the two PhaseNet models at your stations; `results/
  M21_pick_differences.csv`: their differences from your picks.
- `results/pq_picks/`: the pick files of your three PocketQuake runs; `work/.sgtl/runs.jsonl` logs the runs.
- `figures/probability_curves.png` and `figures/model_minus_manual.png`.

Module M30 computes travel times, and module M31 locates the Sangju earthquakes with picks like these (yours, the
models' or PocketQuake's). Module M51 turns PocketQuake's first motions into focal mechanisms. Your cluster
`PQ_SLUG` stays registered; later modules may start from it or stage a new one.